In [ ]:
import sympy as sp

In [ ]:
# define independent variables 
x,y = sp.symbols('x, y', real=True)
xv = sp.Matrix([x,y])

# define the rhs of the governing equation
w = sp.symbols("omega")
u1,u2 = sp.symbols("u_x, u_y", real=True)
u1 = sp.Function("u_x")(x, y)
u2 = sp.Function("u_y")(x, y)
u = sp.Matrix([u1,u2])
grad_w = sp.Matrix([sp.Derivative(w,x), sp.Derivative(w,y)])
FU = -u.dot(grad_w)
FU

-u_x(x, y)*Derivative(omega, x) - u_y(x, y)*Derivative(omega, y)

In [ ]:
# define q(t)
N = 2 #number of vortexes

q = sp.Matrix()

A = sp.symbols("A", real=True)
L = sp.symbols("L", real=True, positive=True)

xc= sp.symbols("x_c", real=True)
yc= sp.symbols("y_c", real=True)

# xc= sp.Matrix()
# yc= sp.Matrix()
# r = sp.Matrix()

# for i in range(N):
#     xc = sp.Matrix([xc, sp.symbols("x_c_"+str(i+1), real=True)])
#     yc = sp.Matrix([yc, sp.symbols("y_c_"+str(i+1), real=True)])
    # r = sp.Matrix([r, sp.symbols("r_"+str(i+1), real=True, positive=True)])
    # r = sp.Matrix([r, sp.Function("r_"+str(i+1))(x, y, xc[i], yc[i])])

q = sp.Matrix([A, L, xc, yc])
# qr = sp.Matrix([A, L, r])

q

Matrix([
[  A],
[  L],
[x_c],
[y_c]])

In [ ]:
# define the ansatz u_hat(x; q)
ansatz_gamma = 0
# for i in range(N):
#     ansatz_gamma = ansatz_gamma + A*sp.exp(-(((x-xc[i])**2+(y-yc[i])**2))/L**2)
ansatz_gamma = A*sp.exp(-(((x-xc)**2+(y-yc)**2))/L**2) + A*sp.exp(-(((x+xc)**2+(y+yc)**2))/L**2)

ansatz_gamma

A*exp((-(x - x_c)**2 - (y - y_c)**2)/L**2) + A*exp((-(x + x_c)**2 - (y + y_c)**2)/L**2)

In [ ]:
ansatz_u = sp.Matrix([
    sp.Derivative(ansatz_gamma,y).doit().simplify(),
    -sp.Derivative(ansatz_gamma, x).doit().simplify()
])

ansatz_u

Matrix([
[-2*A*((y - y_c)*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2) + (y + y_c)*exp(-((x + x_c)**2 + (y + y_c)**2)/L**2))/L**2],
[ 2*A*((x - x_c)*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2) + (x + x_c)*exp(-((x + x_c)**2 + (y + y_c)**2)/L**2))/L**2]])

In [ ]:
ansatz = (- sp.Derivative(ansatz_gamma, x, 2) - sp.Derivative(ansatz_gamma, y, 2)).doit()
ansatz.simplify()

4*A*(L**2*(exp(-((x + x_c)**2 + (y + y_c)**2)/L**2) + exp(-((x - x_c)**2 + (y - y_c)**2)/L**2)) - (x - x_c)**2*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2) - (x + x_c)**2*exp(-((x + x_c)**2 + (y + y_c)**2)/L**2) - (y - y_c)**2*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2) - (y + y_c)**2*exp(-((x + x_c)**2 + (y + y_c)**2)/L**2))/L**4

In [ ]:
# compute partial derivatives du/dqi
dwdq = ansatz.diff(q)

dwdq.simplify()

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 4*(L**2*(exp(-((x + x_c)**2 + (y + y_c)**2)/L**2) + exp(-((x - x_c)**2 + (y - y_c)**2)/L**2)) - (x - x_c)**2*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2) - (x + x_c)**2*exp(-((x + x_c)**2 + (y + y_c)**2)/L**2) - (y - y_c)**2*exp(-((x - x_c)**2 + (y - y_c)**2)/L**2) - (y + y_c)**2*exp(-((x + x_c)**2 + (y + y_c)**2)/L**2))/L**4],
[-8*A*exp(-(x - x_c)**2/L**2 - (y - y_c)**2/L**2)/L**3 - 8*A*exp(-(x + x_c)**2/L**2 - (y + y_c)**2/L**2)/L**3 + 24*A*(x - x_c)**2*exp(-(x - x_c)**2/L**2 - (y - y_c)**2

In [ ]:
xmin,xmax = sp.symbols("x_{min}, x_{max}")
# define the inner product according to the problem
def inner_prod_H(f, g):
    ix = sp.integrate((f*g).expand(),(x, -sp.oo, sp.oo))
    return sp.integrate(ix.expand(),(y, -sp.oo, sp.oo)).expand()

In [ ]:
inner_prod_H(dwdq[0], dwdq[0]).simplify()
# m00 = (dwdq[0]**2).simplify()

# m00i = sp.integrate(m00.expand(), (x, -sp.oo, +sp.oo))

8*pi*(L**4*exp(2*(x_c**2 + y_c**2)/L**2) + L**4 - 4*L**2*(x_c**2 + y_c**2) + 2*x_c**4 + 4*x_c**2*y_c**2 + 2*y_c**4)*exp(-2*(x_c**2 + y_c**2)/L**2)/L**6

In [ ]:
# construct the matrix M_ij = <du/dqi, du/dqj>_H
n = len(q)
M = sp.zeros(n, n)

for i in range(n):
    M[i, i] = inner_prod_H(dwdq[i], dwdq[i]).simplify()
    print(M[i, i])
    for j in range(i+1, n):
        M[i, j] = inner_prod_H(dwdq[i], dwdq[j]).simplify()
        M[j,i] = M[i,j]
        print(M[i, j])


8*pi*(L**4*exp(2*(x_c**2 + y_c**2)/L**2) + L**4 - 4*L**2*(x_c**2 + y_c**2) + 2*x_c**4 + 4*x_c**2*y_c**2 + 2*y_c**4)*exp(-2*(x_c**2 + y_c**2)/L**2)/L**6
8*pi*A*(-L**6*exp(2*(x_c**2 + y_c**2)/L**2) - L**6 + 10*L**4*(x_c**2 + y_c**2) - 14*L**2*(x_c**4 + 2*x_c**2*y_c**2 + y_c**4) + 4*x_c**6 + 12*x_c**4*y_c**2 + 12*x_c**2*y_c**4 + 4*y_c**6)*exp(-2*(x_c**2 + y_c**2)/L**2)/L**9
16*pi*A*x_c*(-3*L**4 + 6*L**2*(x_c**2 + y_c**2) - 2*x_c**4 - 4*x_c**2*y_c**2 - 2*y_c**4)*exp(-2*(x_c**2 + y_c**2)/L**2)/L**8
16*pi*A*y_c*(-3*L**4 + 6*L**2*(x_c**2 + y_c**2) - 2*x_c**4 - 4*x_c**2*y_c**2 - 2*y_c**4)*exp(-2*(x_c**2 + y_c**2)/L**2)/L**8
32*pi*A**2*(L**8*exp(2*(x_c**2 + y_c**2)/L**2) + L**8 - 10*L**6*(x_c**2 + y_c**2) + 20*L**4*(x_c**4 + 2*x_c**2*y_c**2 + y_c**4) - 12*L**2*(x_c**6 + 3*x_c**4*y_c**2 + 3*x_c**2*y_c**4 + y_c**6) + 2*x_c**8 + 8*x_c**6*y_c**2 + 12*x_c**4*y_c**4 + 8*x_c**2*y_c**6 + 2*y_c**8)*exp(-2*(x_c**2 + y_c**2)/L**2)/L**12


KeyboardInterrupt: 

In [ ]:
M

In [ ]:
FU

-u_x(x, y)*Derivative(omega, x) - u_y(x, y)*Derivative(omega, y)

In [ ]:
# compute rhs from the ansatz
Fua = FU.subs(u1, ansatz_u[0]).subs(u2, ansatz_u[1]).subs(w, ansatz).doit()
Fua.simplify()

In [ ]:
# compute f
n = len(q)
f = sp.zeros(n, 1)

for i in range(n):
    f[i] = inner_prod_H(dwdq[i], Fua).simplify()
    print(f[i])

f

In [ ]:
q_dot = M.inv()*f

q_dot.simplify()

In [ ]:
q_dot